In [94]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression, RFE, SelectFromModel, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV


import pandas as pd
import numpy as np

### Extração

In [95]:
df = fetch_california_housing()
display(df.target_names)
print("-"*20)
display(df.feature_names)

['MedHouseVal']

--------------------


['MedInc',
 'HouseAge',
 'AveRooms',
 'AveBedrms',
 'Population',
 'AveOccup',
 'Latitude',
 'Longitude']

### Split

In [96]:
#X = pd.DataFrame(df.data, columns=df.feature_names)
#y = pd.Series(df.target, name=df.target_names[0])

X, y = fetch_california_housing(return_X_y=True, as_frame=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
) # Stratify não entra para targets contínuas

X_train.shape


(16512, 8)

### Análise de Desempenho

In [97]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
model = HistGradientBoostingRegressor(random_state=42)

base = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')

print(base.mean(), base.std())

0.8321651439002877 0.0055347036554319815


### Treinar com menos features

In [98]:
def create_pipe(k: int):
    pipe = Pipeline([
    ("sel", SelectKBest(f_regression, k=k)),
    ("model", model),
    ])

    return pipe

sel = (2,4,6)

sel_2 = cross_val_score(create_pipe(sel[0]), X_train, y_train, cv=cv, scoring='r2')
sel_4 = cross_val_score(create_pipe(sel[1]), X_train, y_train, cv=cv, scoring='r2')
sel_6 = cross_val_score(create_pipe(sel[2]), X_train, y_train, cv=cv, scoring='r2')

print(sel_2.mean(), sel_2.std())
print(sel_4.mean(), sel_4.std())
print(sel_6.mean(), sel_6.std())

0.5465476060928671 0.008435627833003993
0.6677881374851529 0.008388035499614843
0.8180586879434688 0.006921541716512432


### E se trocar o método de seleção?

In [103]:
rf = RandomForestRegressor(random_state=42, n_estimators=200)

def build_pipe(selector):
    return Pipeline([
        ("sc", StandardScaler()),
        ("sel", selector),
        ("model", HistGradientBoostingRegressor(random_state=42)),
    ])

pipes = {
    "kbest_mi": build_pipe(SelectKBest(score_func=mutual_info_regression, k=4)),
    "rfe_lr": build_pipe(RFE(estimator=LinearRegression(), n_features_to_select=4)),
    "lasso": build_pipe(SelectFromModel(LassoCV(cv=5, random_state=42), threshold=-np.inf, max_features=4)),
    "rf": build_pipe(SelectFromModel(rf, threshold=-np.inf, max_features=4)),
}

for name, pipe in pipes.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2")
    print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}")

pipe = pipes["kbest_mi"]

pipe.fit(X_train, y_train)
display(list(pipe[:-1].get_feature_names_out()))
print(pipe.score(X_test, y_test))

kbest_mi: mean=0.8206, std=0.0061
rfe_lr: mean=0.8160, std=0.0073
lasso: mean=0.8160, std=0.0073
rf: mean=0.8226, std=0.0065


['MedInc', 'AveRooms', 'Latitude', 'Longitude']

0.8218177956640933
